<a href="https://colab.research.google.com/github/isali28/HashingLab-Attack-/blob/main/FinalProjectMerging_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 1: Burger King

In [ ]:
import requests
import pandas as pd

base_url = "https://services.arcgis.com/AgwDJMQH12AGieWa/arcgis/rest/services/BurgerKing_Locations/FeatureServer/0/query"

params = {
    "where": "1=1",
    "outFields": "*",
    "f": "json",
    "resultOffset": 0,
    "resultRecordCount": 1000,
    "returnGeometry": "true",
    "outSR": "4326"  # WGS84 lat/lon
}

all_features = []

while True:
    response = requests.get(base_url, params=params)
    data = response.json()
    features = data.get("features", [])

    if not features:
        break

    for feature in features:
        row = feature["attributes"]
        if feature.get("geometry"):
            row["longitude"] = feature["geometry"].get("x")
            row["latitude"] = feature["geometry"].get("y")
        all_features.append(row)

    print(f"Fetched {len(all_features)} records so far...")
    params["resultOffset"] += len(features)

    if not data.get("exceededTransferLimit"):
        break

df = pd.DataFrame(all_features)
df.to_csv("burger_king_locations.csv", index=False)
print(f"\nDone! Saved {len(df)} records to burger_king_locations.csv")

Fetched 1000 records so far...
Fetched 2000 records so far...
Fetched 3000 records so far...
Fetched 4000 records so far...
Fetched 5000 records so far...
Fetched 6000 records so far...
Fetched 6642 records so far...

Done! Saved 6642 records to burger_king_locations.csv


In [ ]:
#extract zipcode from census data csv file

census_df=pd.read_csv("/content/sample_data/ACSDT5Y2024.B19013-Data.csv")
census_df.head()


,GEO_ID,NAME,B19013_001E,B19013_001M,Unnamed: 4
0,Geography,Geographic Area Name,Estimate!!Median household income in the past ...,Margin of Error!!Median household income in th...,NaN
1,860Z200US00601,ZCTA5 00601,19454,1546,NaN
2,860Z200US00602,ZCTA5 00602,21420,1811,NaN
3,860Z200US00603,ZCTA5 00603,20933,1650,NaN
4,860Z200US00606,ZCTA5 00606,20992,3212,NaN


In [ ]:
#let's get rid of every column besides the GEO_ID and the Median Household Income, since we won't be needing those!
census_df=census_df.drop(["B19013_001M","NAME", "Unnamed: 4"], axis=1)
census_df=census_df.iloc[1:]

In [ ]:
#now, let's extract the zipcode!
census_df["ZIP"]=census_df["GEO_ID"].str[-5:]
census_df.head()


,GEO_ID,B19013_001E,ZIP
1,860Z200US00601,19454,00601
2,860Z200US00602,21420,00602
3,860Z200US00603,20933,00603
4,860Z200US00606,20992,00606
5,860Z200US00610,24496,00610


In [ ]:
merged_data=pd.merge(df,census_df, on='ZIP', how='inner')

In [ ]:
merged_data["median household income"]=merged_data["B19013_001E"]
merged_data=merged_data.drop(["B19013_001E"],axis=1)
merged_data.head()

,OBJECTID,LOCNUM,CONAME,STREET,CITY,STATE,STATE_NAME,ZIP,ZIP4,NAICS,...,LOC_NAME,STATUS,SCORE,SOURCE,LON,LAT,longitude,latitude,GEO_ID,median household income
0,1,870357621,BURGER KING,FARRINGTON HWY,WAIANAE,HI,Hawaii,96792,3065,72251117,...,PointAddress,M,100.0,,-158.1837,21.4352,-158.1837,21.4352,860Z200US96792,87509
1,2,581671641,BURGER KING,FARRINGTON HWY,KAPOLEI,HI,Hawaii,96707,2052,72251117,...,PointAddress,M,100.0,,-158.0786,21.3378,-158.0786,21.3378,860Z200US96707,119940
2,3,256405770,BURGER KING,FORT WEAVER RD,EWA BEACH,HI,Hawaii,96706,2246,72251117,...,PointAddress,M,100.0,INFOGROUP,-158.0130,21.3174,-158.0130,21.3174,860Z200US96706,129456
3,4,415204032,BURGER KING,BUILDING 2096,HICKAM AFB,HI,Hawaii,96853,,72251117,...,Postal,M,100.0,INFOGROUP,-157.9516,21.3326,-157.9516,21.3326,860Z200US96853,-
4,5,870325438,BURGER KING,MEHEULA PKWY,MILILANI,HI,Hawaii,96789,1786,72251117,...,StreetAddress,M,100.0,,-158.0078,21.4534,-158.0078,21.4534,860Z200US96789,127066


In [ ]:
merged_data.to_csv("merged_data_burgerking.csv")

In [ ]:
# wendy's data

import kagglehub

# Download latest version
path = kagglehub.dataset_download("rishidamarla/fast-food-restaurants-in-america")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fast-food-restaurants-in-america' dataset.
Path to dataset files: /kaggle/input/fast-food-restaurants-in-america


In [ ]:
import os

# List the contents of the directory to find the CSV file
file_list = os.listdir(path)

# Assuming there's only one CSV file in the directory, or it's named predictably
# We can filter for .csv files
csv_files = [f for f in file_list if f.endswith('.csv')]

# Construct the full path to the CSV file
csv_file_path = os.path.join(path, csv_files[0])
df = pd.read_csv(csv_file_path)


In [ ]:
df["ZIP"]=df["postalCode"]
df.head()

,address,city,country,keys,latitude,longitude,name,postalCode,province,websites,ZIP
0,324 Main St,Massena,US,us/ny/massena/324mainst/-1161002137,44.92130,-74.89021,McDonald's,13662,NY,"http://mcdonalds.com,http://www.mcdonalds.com/...",13662
1,530 Clinton Ave,Washington Court House,US,us/oh/washingtoncourthouse/530clintonave/-7914...,39.53255,-83.44526,Wendy's,43160,OH,http://www.wendys.com,43160
2,408 Market Square Dr,Maysville,US,us/ky/maysville/408marketsquaredr/1051460804,38.62736,-83.79141,Frisch's Big Boy,41056,KY,"http://www.frischs.com,https://www.frischs.com...",41056
3,6098 State Highway 37,Massena,US,us/ny/massena/6098statehighway37/-1161002137,44.95008,-74.84553,McDonald's,13662,NY,"http://mcdonalds.com,http://www.mcdonalds.com/...",13662
4,139 Columbus Rd,Athens,US,us/oh/athens/139columbusrd/990890980,39.35155,-82.09728,OMG! Rotisserie,45701,OH,"http://www.omgrotisserie.com,http://omgrotisse...",45701


In [ ]:
wendys_df=df[df['name']=="Wendy's"]

In [ ]:
wendys_df.head()

,address,city,country,keys,latitude,longitude,name,postalCode,province,websites,ZIP
1,530 Clinton Ave,Washington Court House,US,us/oh/washingtoncourthouse/530clintonave/-7914...,39.532550,-83.445260,Wendy's,43160,OH,http://www.wendys.com,43160
8,205 W Church St,Batesburg,US,us/sc/batesburg/205wchurchst/-791445730,33.913350,-81.533300,Wendy's,29006,SC,http://www.wendys.com,29006
128,1231 Holcomb Blvd,Camp Lejeune,US,us/nc/camplejeune/1231holcombblvd/-791445730,34.724180,-77.344440,Wendy's,28547,NC,http://www.wendys.com,28547
217,313 2nd St,North Wilkesboro,US,us/nc/northwilkesboro/3132ndst/-791445730,36.165620,-81.136580,Wendy's,28659,NC,http://www.wendys.com,28659
245,14493 Gideon Dr,Woodbridge,US,us/va/woodbridge/14493gideondr/-791445730,38.637623,-77.297706,Wendy's,22192,VA,http://www.wendys.com,22192


In [ ]:
merged_data=pd.merge(wendys_df,census_df, on='ZIP', how='inner')
merged_data=merged_data.drop(["B19013_001E","postalCode","name"],axis=1)
merged_data.head()


,address,city,country,keys,latitude,longitude,province,websites,ZIP,GEO_ID
0,530 Clinton Ave,Washington Court House,US,us/oh/washingtoncourthouse/530clintonave/-7914...,39.53255,-83.44526,OH,http://www.wendys.com,43160,860Z200US43160
1,530 Clinton Ave,Washington Court House,US,us/oh/washingtoncourthouse/530clintonave/-7914...,39.53255,-83.44526,OH,http://www.wendys.com,43160,860Z200US43160
2,205 W Church St,Batesburg,US,us/sc/batesburg/205wchurchst/-791445730,33.91335,-81.53330,SC,http://www.wendys.com,29006,860Z200US29006
3,1231 Holcomb Blvd,Camp Lejeune,US,us/nc/camplejeune/1231holcombblvd/-791445730,34.72418,-77.34444,NC,http://www.wendys.com,28547,860Z200US28547
4,313 2nd St,North Wilkesboro,US,us/nc/northwilkesboro/3132ndst/-791445730,36.16562,-81.13658,NC,http://www.wendys.com,28659,860Z200US28659


In [ ]:
merged_data.to_csv("merged_data_wendys.csv")

In [ ]:
sonic_df=df[df['name']=="Sonic"]
merged_data=pd.merge(sonic_df,census_df, on='ZIP', how='inner')
merged_data=merged_data.drop(["B19013_001E","postalCode","name"],axis=1)
merged_data.head()




,address,city,country,keys,latitude,longitude,province,websites,ZIP,GEO_ID
0,8841 N Highway 146,Baytown,US,us/tx/baytown/8841nhighway146/109620780,29.820624,-94.899152,TX,https://locations.sonicdrivein.com/tx/baytown/...,77523,860Z200US77523
1,823 W Base St,Madison,US,us/fl/madison/823wbasest/109620780,30.469651,-83.420237,FL,NaN,32340,860Z200US32340
2,1250 Farm To Market 802,Brownsville,US,us/tx/brownsville/1250farmtomarket802/109620780,25.949460,-97.500754,TX,http://sonicdrivein.com,78526,860Z200US78526
3,1000 Woodstock Dr,Clarksville,US,us/in/clarksville/1000woodstockdr/109620780,38.324974,-85.763146,IN,http://sonicdrivein.com,47129,860Z200US47129
4,216 S Broadway St,La Porte,US,us/tx/laporte/216sbroadwayst/109620780,29.663423,-95.019624,TX,http://sonicdrivein.com,77571,860Z200US77571


In [ ]:
merged_data.to_csv("merged_data_sonic.csv")